# Stage 4: Inflation Forecasting & MPC Policy Classification (AI Layer)
This notebook implements the machine learning layers of the project:
1. **Core Inflation Forecasting (Prophet):** Predicts the Core CPI YoY series 6 months out to capture seasonal patterns and outline future trends with 90% confidence bands.
2. **MPC Policy Action Classifier (XGBoost):** Classifies monetary policy decisions into *Rate Hike* vs *Hold/Cut* using inflation lag features, validated using TimeSeriesSplit.
3. **Explainable AI (SHAP):** Explains model predictions using SHAP feature importances and single-decision force plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import xgboost as xgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix
import os

# Load cleaned CPI data
df_cpi = pd.read_csv('../data/cleaned_cpi.csv')
df_cpi['date'] = pd.to_datetime(df_cpi['date'])

# Load processed MPC decisions dataset
df_mpc = pd.read_csv('../data/processed_cpi_mpc.csv')
df_mpc['date'] = pd.to_datetime(df_mpc['date'])

os.makedirs('../outputs', exist_ok=True)

## 1. Core CPI Inflation Forecasting (Prophet)
Prophet is ideal for Indian macroeconomic indicators because it handles seasonal monsoon effects and crop harvest cycles automatically. We use an additive seasonality model with a 90% confidence interval.

In [ ]:
# Prophet requires columns 'ds' (datestamp) and 'y' (target value)
core_prophet = df_cpi[['date', 'cpi_core_yoy']].rename(
    columns={'date': 'ds', 'cpi_core_yoy': 'y'}
).dropna()

# Initialize and fit Prophet model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive',  # CPI components represent additive YoY rates
    interval_width=0.90           # 90% uncertainty/confidence interval
)

model.fit(core_prophet)

# Generate future date dataframe for the next 6 months
future = model.make_future_dataframe(periods=6, freq='MS')
forecast = model.predict(future)

# Plot 1: 6-month forecast with confidence bands
fig1 = model.plot(forecast)
fig1.suptitle('Prophet Forecast: India Core CPI (YoY %) — Next 6 Months', fontsize=11, y=1.02)
plt.savefig('../outputs/04_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Seasonal decomposition components
fig2 = model.plot_components(forecast)
plt.savefig('../outputs/05_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the forecasted rates for the next 6 months
print("Forecasted values for the next 6 months (yhat = predicted, yhat_lower/upper = bounds):")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(6).round(2).to_string(index=False))

## 2. MPC Policy Action Classifier (XGBoost)
We build a binary classification model to predict whether the MPC will hike rates (`is_hike = 1`) or not (`is_hike = 0`).

### Feature Selection:
* Inflation levels: `cpi_general_yoy`, `cpi_food_yoy`, `cpi_fuel_yoy`, `cpi_core_yoy`
* Inflation trends: `core_lag1` (previous meeting), `core_lag2` (two meetings ago), `core_3m_avg`

### Cross-Validation Strategy:
* For time-series data, random split introduces lookahead bias. We use `TimeSeriesSplit` (5 splits) to validate the classifier sequentially over historical periods.

In [ ]:
# Columns used as predictors
feature_cols = [
    'cpi_general_yoy', 'cpi_food_yoy', 'cpi_fuel_yoy', 'cpi_core_yoy',
    'core_lag1', 'core_lag2', 'core_3m_avg'
]

# Filter out rows with missing lags
model_df = df_mpc[feature_cols + ['is_hike']].dropna().copy()
X = model_df[feature_cols]
y = model_df['is_hike']

# Set up Time Series Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

print("=== Time-Series Cross-Validation Accuracy ===")
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    clf = xgb.XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42
    )
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    cv_scores.append(score)
    print(f"Fold {fold + 1} Test Accuracy: {score:.3f}")

print(f"Average Time-Series CV Accuracy: {np.mean(cv_scores):.3f}")
print(f"(Note: Dataset has {len(model_df)} MPC meetings. Results should be interpreted as pattern identifiers rather than rigid forecasters.)")

## 3. Explainable AI (SHAP Explainer)
We train a final model on the complete historical dataset and fit a SHAP tree explainer to check feature contributions. This tells us what macro indicators most strongly influence MPC hikes.

In [ ]:
# Train final model
clf_final = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)
clf_final.fit(X, y)

# Fit KernelExplainer with a lambda wrapper to avoid XGBoost 3.0+ compatibility bugs
print("Initializing KernelExplainer...")
background = X.median().to_frame().T
explainer = shap.KernelExplainer(lambda x: clf_final.predict_proba(x), background)

print("Computing SHAP values...")
shap_values = explainer.shap_values(X)
print("SHAP values computed successfully!")

# Extract class-1 SHAP values (rate hikes)
# KernelExplainer returns a 3D array of shape (n_samples, n_features, 2)
pos_shap_values = shap_values[:, :, 1]
expected_val = explainer.expected_value[1]

# Plot 1: SHAP Feature Importance Bar Summary
plt.figure(figsize=(10, 6))
shap.summary_plot(
    pos_shap_values, 
    X, 
    feature_names=feature_cols,
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance: What Drives RBI Rate Hike Predictions?', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('../outputs/06_shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Single Decision Force Plot
We explain a specific decision (e.g. the first post-COVID hike on May 4, 2022) to trace which macro features pushed the probability toward a hike.

In [ ]:
# Locate the row for the May 4, 2022 meeting
hike_idx = df_mpc[df_mpc['date'] == '2022-05-04'].index

if len(hike_idx) > 0:
    idx_val = hike_idx[0]
    # Find positional index inside the X matrix
    x_idx = model_df.index.get_loc(idx_val)
    
    print(f"Explaining rate hike decision on date: {df_mpc.loc[idx_val, 'date'].strftime('%Y-%m-%d')}")
    print("Features at meeting:")
    print(X.iloc[x_idx].round(2))
    
    # Render force plot for this decision
    shap.initjs() # initialize javascript visualization
    fig = shap.force_plot(
        expected_val, 
        pos_shap_values[x_idx], 
        X.iloc[x_idx], 
        matplotlib=True, 
        show=False
    )
    plt.title('SHAP Force Plot: Deconstructing the May 4, 2022 Rate Hike Decision', fontsize=11, pad=20)
    plt.savefig('../outputs/07_shap_single_decision.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Hike date 2022-05-04 not found in features index.")